In [ ]:
# %% [markdown]
# # ML-модель важности навыков: обучение, оценка, рекомендации

# %%
import sys
import os
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from src.predictors.ltr_recommendation_engine import LTRRecommendationEngine
from src.config import DATA_RAW_DIR, MODELS_DIR, STUDENTS_DIR
from src.parsing.vacancy_parser import VacancyParser
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# %%
# 1. Загрузка сырых вакансий (как в main.py)
raw_file = DATA_RAW_DIR / "hh_vacancies_basic.json"
if not raw_file.exists():
    raise FileNotFoundError("Сначала запустите сбор вакансий: python main.py --it-sector")

with open(raw_file, 'r', encoding='utf-8') as f:
    vacancies = json.load(f)
print(f"Загружено {len(vacancies)} сырых вакансий")

# %%
# 2. Обучение модели
engine = LTRRecommendationEngine()
engine.fit(vacancies, test_size=0.2)
print("Модель обучена и сохранена в", engine.model_path)

# %%
# 3. Метрики и графики обучения
# (уже сохранены в MODELS_DIR: confusion_matrix.png, roc_curve.png для регрессии pred_vs_actual.png, residuals_dist.png)
print("Графики качества модели сохранены в:", MODELS_DIR)

# %%
# 4. Важность признаков
importance_df = pd.DataFrame({
    'feature': engine.feature_names,
    'importance': engine.pipeline.named_steps['clf'].feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x='importance', y='feature')
plt.title('Топ-10 важных признаков модели')
plt.show()

# %%
# 5. Демонстрация рекомендаций для профиля 'base'
student_file = STUDENTS_DIR / "base_competency.json"
if student_file.exists():
    with open(student_file, 'r', encoding='utf-8') as f:
        student_data = json.load(f)
        student_skills = student_data.get("навыки", [])
else:
    student_skills = ["python", "sql"]  # fallback

market_skills = list(engine.skill_metadata.keys())
missing = [s for s in market_skills if s not in student_skills]

recs = engine.predict_skill_impact(student_skills, missing)
print("Рекомендации для профиля 'base':")
for skill, score, expl in recs[:10]:
    print(f"  {skill:<20} важность: {score:.1f}% — {expl}")

# %%
# 6. Сравнение с простой сортировкой по частоте
top_freq = sorted(engine.skill_metadata.items(), key=lambda x: x[1]['frequency'], reverse=True)[:10]
print("\nТоп-10 по частоте (без ML):")
for skill, meta in top_freq:
    print(f"  {skill:<20} частота: {meta['frequency']}")

# Визуализация сравнения
freq_skills = [s for s, _ in top_freq]
ml_skills = [s for s, _, _ in recs[:10]]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.barplot(x=[meta['frequency'] for _, meta in top_freq], y=freq_skills)
plt.title('Топ по частоте')
plt.subplot(1, 2, 2)
sns.barplot(x=[score for _, score, _ in recs[:10]], y=ml_skills)
plt.title('ML-рекомендации')
plt.tight_layout()
plt.show()

Загружено 300 сырых вакансий


❌ Не удалось загрузить модель эмбеддингов: Файл подкачки слишком мал для завершения операции. (os error 1455)
Нет текстов для BM25 (все вакансии пусты)
BM25 вернул пустой результат → возвращаем fallback


Модель обучена и сохранена в c:\Users\максим\workvs\compare_competencies\data\models\skill_importance_regressor.joblib
Графики качества модели сохранены в: c:\Users\максим\workvs\compare_competencies\data\models


KeyError: 'clf'